In [1]:
import torch
import torch.nn as nn
import torch.optim as optim

In [2]:
import torchvision
from torchvision.datasets import CIFAR10

In [3]:
from torch.utils.data import DataLoader
import torchvision.transforms as transforms

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,0.5,0.5),(0.5,0.5,0.5))
])

trainset = CIFAR10(root="./data/",download=True, train=True,transform=transform)
testset = CIFAR10(root="./data/",download=True, train=False, transform=transform)

In [4]:
train_dataloader = DataLoader(trainset, batch_size=64, shuffle=True)
test_dataloader = DataLoader(testset, batch_size=64)

In [5]:
class CNN(nn.Module):
    def __init__(self):
        super(CNN, self).__init__()

        self.conv_layer = nn.Sequential(
            nn.Conv2d(3,32,kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2,2),

            nn.Conv2d(32,64,kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2,2),

            nn.Conv2d(64,128,kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2,2)
        )
        self.fc_layer = nn.Sequential(
            nn.Linear(4*4*128, 256),
            nn.ReLU(),

            nn.Linear(256, 10)
        )
    def forward(self, x):
        x = self.conv_layer(x)
        x = x.view(x.size(0),-1)
        x = self.fc_layer(x)

        return x

In [6]:
model = CNN()
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters())

In [7]:
train_loss = []
val_loss = []

epochs = 10
for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    for xb, yb in train_dataloader:
        optimizer.zero_grad()
        outputs = model.forward(xb)
        loss = criterion(outputs, yb)
        loss.backward()
        optimizer.step()
        running_loss+=loss.item()
    epoch_train_loss = running_loss/len(train_dataloader)
    train_loss.append(epoch_train_loss)
    print(f"{epoch+1}/{epochs}--->train loss = {epoch_train_loss}")

1/10--->train loss = 1.3938215192016739
2/10--->train loss = 0.9491277316494671
3/10--->train loss = 0.7580027023087377
4/10--->train loss = 0.6315125709451983
5/10--->train loss = 0.5268185840886267
6/10--->train loss = 0.43431458507886017
7/10--->train loss = 0.3507155041922541
8/10--->train loss = 0.27663550507801266
9/10--->train loss = 0.21669403629858627
10/10--->train loss = 0.17376132297050922


In [8]:
correct_labels = 0
total_labels = 0

model.eval()
with torch.no_grad():
    for xb, yb in test_dataloader:
        outputs = model.forward(xb)
        _, predicted = torch.max(outputs,1 )
        correct_labels+=(predicted==yb).sum().item()
        total_labels+=yb.size(0)
print(f"accuracy = {correct_labels/total_labels}")

accuracy = 0.7535
